In [1]:
from dataAnalysis.DataAnalysis import DataAnalysis
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

data = pd.read_csv(r"extdata/sbcdata.csv", header=0)
data_analysis = DataAnalysis(data)

/home/dwalke/git/sbc_app/dataAnalysis/data/Filter.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['Label'] = self.data['Diagnosis']
/home/dwalke/git/sbc_app/dataAnalysis/data/Filter.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['Label'] = self.data['Diagnosis']
/home/dwalke/git/sbc_app/dataAnalysis/data/Filter.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cave

In [2]:
sorted_train_data = data_analysis.get_training_data().sort_values(by="Id").reset_index(drop = True)
train_df = sorted_train_data.loc[:sorted_train_data.shape[0]*.8,:]
val_df = sorted_train_data.loc[sorted_train_data.shape[0]*.8:,:]
test_df = data_analysis.get_testing_data()
gw_df = data_analysis.get_gw_testing_data()

In [3]:
import numpy as np
import torch
from dataAnalysis.Constants import FEATURES, LABEL_COLUMN_NAME

def convert_cat_feature(df):
    df = df.copy()
    df["SexCategory"] = (df["Sex"] == "W").astype(int)
    return df
    
def get_graph(df):
    edge_index = []
    df = convert_cat_feature(df)
    df = df.sort_values(by=["Id", "Time"]).reset_index(drop=True)
    
    ## group df by ids
    for identifier, group in df.groupby("Id"):
        offset = group.index[0]
        triu_matrix = np.triu((group.index.values + np.identity(1))[0])
        triu_exp_matrix = np.expand_dims(triu_matrix, axis=-1)
    
        idx_shape = group.index.shape[0]
        idx_matrix = np.ones((idx_shape, idx_shape)) * np.arange(idx_shape) + 1 + offset
        idx_matrix = np.transpose(idx_matrix)
        idx_exp_matrix = np.expand_dims(idx_matrix, axis=-1)
    
        unprocess_edges = np.concatenate((idx_exp_matrix, triu_exp_matrix), axis=-1)
        reshaped_unprocess_edges = np.reshape(unprocess_edges, (-1, 2))
        mask = (reshaped_unprocess_edges[:, 0] * reshaped_unprocess_edges[:, 1]) != 0
        edge_index.append((reshaped_unprocess_edges[mask] - 1).astype(np.int64))
    edge_index_torch = torch.from_numpy(np.concatenate(edge_index)).type(torch.long).transpose(0,1)
    features_torch = torch.from_numpy(df[FEATURES].values).type(torch.float)
    labels_torch = torch.from_numpy((df.sort_values(by=["Id", "Time"])[LABEL_COLUMN_NAME] == "Sepsis").values).type(torch.long)
    return features_torch, edge_index_torch, labels_torch

In [4]:
X_train_comp, edge_index_train_comp, y_train_comp = get_graph(sorted_train_data)
X_train, edge_index_train, y_train = get_graph(train_df)
X_val, edge_index_val, y_val = get_graph(val_df)
X_test, edge_index_test, y_test = get_graph(test_df)
X_gw, edge_index_gw, y_gw = get_graph(gw_df)

In [5]:
rev_edge_index_train_comp = torch.zeros_like(edge_index_train_comp)
rev_edge_index_train_comp[0,:] = edge_index_train_comp[1,:]
rev_edge_index_train_comp[1,:] = edge_index_train_comp[0,:]

rev_edge_index_train = torch.zeros_like(edge_index_train)
rev_edge_index_train[0,:] = edge_index_train[1,:]
rev_edge_index_train[1,:] = edge_index_train[0,:]

rev_edge_index_val = torch.zeros_like(edge_index_val)
rev_edge_index_val[0,:] = edge_index_val[1,:]
rev_edge_index_val[1,:] = edge_index_val[0,:]

rev_edge_index_test = torch.zeros_like(edge_index_test)
rev_edge_index_test[0,:] = edge_index_test[1,:]
rev_edge_index_test[1,:] = edge_index_test[0,:]

rev_edge_index_gw = torch.zeros_like(edge_index_gw)
rev_edge_index_gw[0,:] = edge_index_gw[1,:]
rev_edge_index_gw[1,:] = edge_index_gw[0,:]

In [6]:
from torch_geometric.utils import to_undirected

undir_edge_index_comp = to_undirected(edge_index_train_comp)
undir_edge_index_train = to_undirected(edge_index_train)
undir_edge_index_val = to_undirected(edge_index_val)
undir_edge_index_test = to_undirected(edge_index_test)
undir_edge_index_gw = to_undirected(edge_index_gw)

/home/dwalke/.local/lib/python3.10/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


In [7]:
from EnsembleFramework import Framework
from sklearn.ensemble import RandomForestClassifier

def user_function(kwargs):
    return kwargs["updated_features"] - kwargs["mean_neighbors"]

rev_diff_params = {'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': {0: 0.0015, 1: 1},
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': 50,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 0.0005137904629755657,
 'min_samples_split': 0.006603597209362648,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 1000,
 'n_jobs': -1,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

clf = RandomForestClassifier(class_weight="balanced", max_leaf_nodes=79,
                           min_samples_leaf=0.0001,
                           min_samples_split=0.0055,
                           n_estimators=500, random_state=42, n_jobs=-1)
hops = [0, 1]
framework = Framework(hops_list= hops,
                      clfs=[None for _  in hops],
                      attention_configs=[None for i in hops],
                      handle_nan=0.0,
                      gpu_idx=0,
                      user_functions=[user_function for i in hops]
                      )

In [8]:
def get_features(framework, X, edge_index):
    return framework.get_features(X, edge_index, torch.ones(X.shape[0]).type(torch.bool))

In [ ]:
features_train = torch.cat(get_features(framework, X_train_comp, rev_edge_index_train), dim = 1)

In [166]:
clf.fit(features_train.cpu(), y_train_comp)

RandomForestClassifier(class_weight='balanced', max_leaf_nodes=79,
                       min_samples_leaf=0.0001, min_samples_split=0.0055,
                       n_estimators=500, n_jobs=-1, random_state=42)

In [9]:
dir_sets = [("train_comp", X_train_comp, edge_index_train_comp, y_train_comp), ("train", X_train, edge_index_train, y_train), ("val", X_val, edge_index_val, y_val), ("test", X_test, edge_index_test, y_test),
       ("gw", X_gw, edge_index_gw, y_gw)]
dir_sets_dict = {dir_set[0]: (dir_set[1:]) for dir_set in dir_sets}
rev_dir_sets = [("train_comp", X_train_comp, rev_edge_index_train_comp, y_train_comp), ("train", X_train, rev_edge_index_train, y_train), ("val", X_val, rev_edge_index_val, y_val), ("test", X_test, rev_edge_index_test, y_test),
       ("gw", X_gw, rev_edge_index_gw, y_gw)]
rev_dir_sets_dict = {rev_dir_set[0]: (rev_dir_set[1:]) for rev_dir_set in rev_dir_sets}
undir_sets = [("train_comp", X_train_comp, undir_edge_index_comp, y_train_comp), ("train", X_train, undir_edge_index_train, y_train), ("val", X_val, undir_edge_index_val, y_val), ("test", X_test, undir_edge_index_test, y_test),
       ("gw", X_gw, undir_edge_index_gw, y_gw)]
undir_sets_dict = {undir_set[0]: (undir_set[1:]) for undir_set in undir_sets}

In [10]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
def test_auroc_and_auprc(framework, clf, X, edge_index,y):
    features = torch.cat(get_features(framework, X, edge_index), dim = 1)
    pred_proba = clf.predict_proba(features.cpu())[:,1]
    auroc = roc_auc_score(y, pred_proba)

    precision, recall, thresholds = precision_recall_curve(y, pred_proba)
    auprc = auc(recall, precision)
    return auroc, auprc


In [11]:
def eval_rev(framework, clf):
    eval_dict = dict()
    for name in rev_dir_sets_dict:
        X, edge_index,y = rev_dir_sets_dict[name]
        auroc, auprc = test_auroc_and_auprc(framework, clf, X, edge_index,y)
        eval_dict[name] = dict()
        eval_dict[name]["AUROC"] = np.round(auroc, 4)
        eval_dict[name]["AUPRC"] = np.round(auprc, 4)
    return eval_dict
        
def eval_dir(framework, clf):
    eval_dict = dict()
    for name in dir_sets_dict:
        X, edge_index,y = dir_sets_dict[name]
        auroc, auprc = test_auroc_and_auprc(framework,clf, X, edge_index,y)
        eval_dict[name] = dict()
        eval_dict[name]["AUROC"] = np.round(auroc, 4)
        eval_dict[name]["AUPRC"] = np.round(auprc, 4)
    return eval_dict

In [13]:
eval_rev(framework, clf)

NotFittedError: This RandomForestClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [12]:
from hyperopt import fmin, tpe, hp,STATUS_OK, SparkTrials, space_eval 

class SparkTune():
    def __init__(self, clf,user_function,hops,attention_config, y_train, X_train, train_edge_index, y_val, X_val, val_edge_index, parallelism=1):
        self.clf = clf
        self.user_function = user_function
        self.hops = hops
        self.attention_config = attention_config
        self.parallelism = parallelism
        self.y_train = y_train
        self.X_train= X_train
        self.train_edge_index = train_edge_index

        self.y_val = y_val
        self.X_val= X_val
        self.val_edge_index = val_edge_index

        
        
    def objective(self, params):
        framework = Framework(user_functions=[self.user_function for i in self.hops], 
                     hops_list=self.hops,
                     clfs=[None for i in self.hops],
                     gpu_idx=0,
                     handle_nan=0.0,
                    attention_configs=[self.attention_config for i in self.hops])
        features_train = torch.cat(get_features(framework, self.X_train, self.train_edge_index), dim = 1)
        features_val = torch.cat(get_features(framework, self.X_val, self.val_edge_index), dim = 1)
            
        if "n_estimators" in params:
            params["n_estimators"] = int(params["n_estimators"])
        if "max_depth" in params:
            params["max_depth"] = int(params["max_depth"])
        if "max_leaf_nodes" in params:
            params["max_leaf_nodes"] = int(params["max_leaf_nodes"])
        model = self.clf(**params)
        
        model.fit(features_train.cpu(), self.y_train)
        
        y_pred_proba = model.predict_proba(features_val.cpu())[:, 1]
        score = roc_auc_score(self.y_val, y_pred_proba)
        return {'loss': -score, 'status': STATUS_OK}
    
    def search(self, space, max_evals):
        spark_trials = SparkTrials(parallelism = self.parallelism)
        best_params = fmin(self.objective, space, algo=tpe.suggest, trials=spark_trials, max_evals=max_evals, verbose = True)
        return space_eval(space, best_params)

In [13]:
from hyperopt import hp

control_weight = y_train.sum() / y_train.numel()
control_weight_scaled = (y_train.sum()*3) / (y_train.numel()*2)
rf_choices = {
    'class_weight': ["balanced", 
                     "balanced_subsample",
                     {0: control_weight, 1:1},
                     {0: control_weight_scaled, 1:1}
                    ],
    "random_state":  [42],
    "n_jobs":  [-1],
}
rf_space = {
    **{key: hp.choice(key, value) for key, value in rf_choices.items()},
    # 'min_samples_leaf': hp.uniform('min_samples_leaf', 0.00005, 0.001),
    'min_samples_leaf': hp.uniform('min_samples_leaf', 1e-5, 1e-3),
    'min_samples_split': hp.uniform('min_samples_split', 1e-3, 1e-2),
    'n_estimators': hp.qloguniform('n_estimators', low=np.log(100), high=np.log(1_000), q=100),
    'max_leaf_nodes': hp.qloguniform('max_leaf_nodes', low=np.log(50), high=np.log(100), q=1)
}


In [15]:
edge_type_sets = {
    "dir": dir_sets_dict,
    "rev_dir": rev_dir_sets_dict,
    # "undir": undir_sets_dict,
}

In [ ]:
from tqdm.notebook import tqdm

res_dict = dict()
for edge_type_set_name in tqdm(edge_type_sets):
    best_val = 0
    
    res_dict[edge_type_set_name] = dict()
    print("Find best hyperparams")
    X_train, edge_index_train, y_train = edge_type_sets[edge_type_set_name]["train"]
    X_val, edge_index_val, y_val = edge_type_sets[edge_type_set_name]["val"]
    spark_tune = SparkTune(RandomForestClassifier,user_function,[0,1],None, y_train, X_train, edge_index_train, y_val, X_val, edge_index_val, 3)
    params = spark_tune.search(rf_space,20)
    if "n_estimators" in params:
            params["n_estimators"] = int(params["n_estimators"])
    if "max_depth" in params:
        params["max_depth"] = int(params["max_depth"])
    if "max_leaf_nodes" in params:
        params["max_leaf_nodes"] = int(params["max_leaf_nodes"])
    
    framework = Framework(user_functions=[user_function,user_function], 
                     hops_list=[0, 1],
                     clfs=[_, _],
                     gpu_idx=0,
                     handle_nan=0.0,
                    attention_configs=[None, None])
    print("Retrain with best params")
    X_train_comp, edge_index_train_comp, y_train_comp = edge_type_sets[edge_type_set_name]["train_comp"]
    features_train = torch.cat(get_features(framework, X_train_comp, edge_index_train_comp), dim = 1)
    model = RandomForestClassifier(**params)
    model.fit(features_train.cpu(), y_train_comp)

    print("Evaluate")
    eval_dict = dict()
    for name in edge_type_sets[edge_type_set_name]:
        X, edge_index,y = edge_type_sets[edge_type_set_name][name]
        auroc, auprc = test_auroc_and_auprc(framework,model, X, edge_index,y)
        eval_dict[name] = dict()
        eval_dict[name]["AUROC"] = np.round(auroc, 4)
        eval_dict[name]["AUPRC"] = np.round(auprc, 4)
    res_dict[edge_type_set_name] = dict()
    res_dict[edge_type_set_name]["best_params"] = params
    res_dict[edge_type_set_name]["eval_dict"] = eval_dict

In [ ]:
import pandas as pd
for key in res_dict:
    print(key)
    print(pd.DataFrame(res_dict[key]["eval_dict"]))
    print(pd.DataFrame(res_dict[key]["eval_dict"]))
    print(res_dict[key]["best_params"])

In [16]:
### Random Forest
       train_comp   train     val    test      gw
AUROC      0.9343  0.9340  0.9352  0.8889  0.8405
AUPRC      0.0340  0.0334  0.0365  0.0221  0.0091
       train_comp   train     val    test      gw
AUROC      0.9343  0.9340  0.9352  0.8889  0.8405
AUPRC      0.0340  0.0334  0.0365  0.0221  0.0091
{'class_weight': {0: tensor(0.0022), 1: 1}, 'max_leaf_nodes': 96, 'min_samples_leaf': 0.0003683813065159676, 'min_samples_split': 0.0065797321149668915, 'n_estimators': 100, 'n_jobs': -1, 'random_state': 42}
rev_dir
       train_comp   train     val    test      gw
AUROC      0.9642  0.9642  0.9645  0.9425  0.9527
AUPRC      0.0600  0.0596  0.0638  0.0433  0.0284
       train_comp   train     val    test      gw
AUROC      0.9642  0.9642  0.9645  0.9425  0.9527
AUPRC      0.0600  0.0596  0.0638  0.0433  0.0284
{'class_weight': {0: tensor(0.0022), 1: 1}, 'max_leaf_nodes': 76, 'min_samples_leaf': 0.0001679711412637778, 'min_samples_split': 0.005791441918035724, 'n_estimators': 400, 'n_jobs': -1, 'random_state': 42}
undir
       train_comp   train     val    test      gw
AUROC      0.9499  0.9502  0.9500  0.9030  0.8834
AUPRC      0.0606  0.0594  0.0666  0.0402  0.0357
       train_comp   train     val    test      gw
AUROC      0.9499  0.9502  0.9500  0.9030  0.8834
AUPRC      0.0606  0.0594  0.0666  0.0402  0.0357
{'class_weight': {0: tensor(0.0022), 1: 1}, 'max_leaf_nodes': 92, 'min_samples_leaf': 0.0001523464471642847, 'min_samples_split': 0.005985838170300448, 'n_estimators': 400, 'n_jobs': -1, 'random_state': 42}


In [58]:
from torch.nn.functional import normalize

def user_function(kwargs):
    return kwargs["updated_features"] - kwargs["mean_neighbors"]
    
random_forest = RandomForestClassifier(**{'class_weight': {0: 0.0022, 1: 1}, 'max_leaf_nodes': 92, 'min_samples_leaf': 0.0001523464471642847, 'min_samples_split': 0.005985838170300448, 'n_estimators': 400, 'n_jobs': -1, 'random_state': 42})
hops = [0, 1]
framework = Framework(user_functions=[user_function, user_function], 
                         hops_list=hops,
                         clfs=[None, None],
                         gpu_idx=0,
                         handle_nan=0.0,
                        attention_configs=[ {'inter_layer_normalize': False,
    'use_pseudo_attention': True,
    'cosine_eps': None,
    'dropout_attn': None} for i in hops])
X_train, edge_index_train, y_train = edge_type_sets["dir"]["train"]
features_train = torch.cat(get_features(framework, X_train, edge_index_train), dim = 1)
print(features_train.cpu().shape)
random_forest.fit(features_train.cpu(), y_train)

print("Evaluate")
eval_dict = dict()
for name in edge_type_sets["dir"]:
    X, edge_index,y = edge_type_sets["dir"][name]
    features = torch.cat(get_features(framework, X, edge_index), dim = 1)
    print(features.cpu().shape)
    pred_proba = random_forest.predict_proba(features.cpu())[:,1]
    auroc = roc_auc_score(y, pred_proba)

    precision, recall, thresholds = precision_recall_curve(y, pred_proba)
    auprc = auc(recall, precision)
    print(name)
    print(auroc)
    print(auprc)


torch.Size([812060, 14])
Evaluate
torch.Size([1015074, 14])
train_comp
0.9465103171485661
0.05934886830794713
torch.Size([812060, 14])
train
0.9552358249196741
0.06085941690034645
torch.Size([203014, 14])
val
0.9144737545872429
0.05452987604885383
torch.Size([366284, 14])
test
0.9060699213664699
0.04273255386953015
torch.Size([438077, 14])
gw
0.8872398297742738
0.03647580840350858


In [34]:
from torch.nn.functional import normalize

def user_function(kwargs):
    return kwargs["updated_features"] - kwargs["mean_neighbors"]
    
random_forest = RandomForestClassifier(class_weight={0:0.0022, 1: 1}, max_leaf_nodes=79,
                           min_samples_leaf=0.0001,
                           min_samples_split=0.0055,
                           n_estimators=500, random_state=42, n_jobs=-1)
hops = [0, 1]
framework = Framework(user_functions=[user_function, user_function], 
                         hops_list=hops,
                         clfs=[None, None],
                         gpu_idx=0,
                         handle_nan=0.0,
                        attention_configs=[None for i in hops])
X_train, edge_index_train, y_train = edge_type_sets["dir"]["train"]
features_train = torch.cat(get_features(framework, X_train, edge_index_train), dim = 1)
print(features_train.cpu().shape)
random_forest.fit(features_train.cpu(), y_train)

print("Evaluate")
eval_dict = dict()
for name in edge_type_sets["dir"]:
    X, edge_index,y = edge_type_sets["dir"][name]
    features = torch.cat(get_features(framework, X, edge_index), dim = 1)
    print(features.cpu().shape)
    pred_proba = random_forest.predict_proba(features.cpu())[:,1]
    auroc = roc_auc_score(y, pred_proba)

    precision, recall, thresholds = precision_recall_curve(y, pred_proba)
    auprc = auc(recall, precision)
    print(name)
    print(auroc)
    print(auprc)


torch.Size([812060, 14])
Evaluate
torch.Size([1015074, 14])
train_comp
0.929081173917625
0.03647843527476746
torch.Size([812060, 14])
train
0.938233145247653
0.03837272695841996
torch.Size([203014, 14])
val
0.8952722618108886
0.03189387094331827
torch.Size([366284, 14])
test
0.8880738774238162
0.02346749109022203
torch.Size([438077, 14])
gw
0.8389663492690971
0.008959297090818052
